In [ ]:
from langchain_classic.prompts import ChatPromptTemplate,PromptTemplate
from langchain_classic.prompts import MessagesPlaceholder, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import ServerlessSpec,Pinecone
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough,RunnableLambda


In [35]:
load_dotenv()

True

In [36]:
pc=Pinecone(api_key=()

In [37]:
index_name = "practice" 

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [38]:
file_path = "D:\ProdRAG\prodRAG\Blockchain_Course_Proposal.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()
print(type(loader))

<class 'langchain_community.document_loaders.pdf.PyPDFLoader'>


In [39]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

In [40]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = HuggingFaceEmbeddings(model_name=model_name)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1660.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [41]:
vectorstore = PineconeVectorStore.from_documents(
    texts,
    embedder,
    index_name=index_name
)

In [50]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
query = "What is the module4?"
results = retriever.invoke(query)

In [51]:
print(results)

[Document(id='aef2e52a-64c1-482f-b011-baf05024b67a', metadata={'author': 'python-docx', 'creationdate': '2025-10-10T09:37:45+05:30', 'creator': 'Microsoft® Word LTSC', 'moddate': '2025-10-10T09:37:45+05:30', 'page': 0.0, 'page_label': '1', 'producer': 'Microsoft® Word LTSC', 'source': 'D:\\ProdRAG\\prodRAG\\Blockchain_Course_Proposal.pdf', 'total_pages': 4.0}, page_content='Module 4: Smart Contracts and Ethereum — Writing and deploying smart contracts using'), Document(id='e9232faa-4b61-493e-a5be-2e83518e5c8b', metadata={'author': 'python-docx', 'creationdate': '2025-10-10T09:37:45+05:30', 'creator': 'Microsoft® Word LTSC', 'moddate': '2025-10-10T09:37:45+05:30', 'page': 0.0, 'page_label': '1', 'producer': 'Microsoft® Word LTSC', 'source': 'D:\\ProdRAG\\prodRAG\\Blockchain_Course_Proposal.pdf', 'total_pages': 4.0}, page_content='blockchain technology. \nModule 2: Cryptography Fundamentals — Basics of hashing, public and private key \nencryption, and digital signatures. \nModule 3: Cons

In [61]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [62]:
rag_prompt = PromptTemplate.from_template(
   template= """You are a helpful assistant.
Use only the provided context to answer the question.
If the answer is not present in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

In [63]:
rag_chain=RunnableParallel({
    "question" :RunnablePassthrough(),
    "context" :retriever| RunnableLambda(format_docs)
}
)

In [64]:
output=rag_chain.invoke(query)

In [66]:
print(output['context'])

Module 4: Smart Contracts and Ethereum — Writing and deploying smart contracts using

blockchain technology. 
Module 2: Cryptography Fundamentals — Basics of hashing, public and private key 
encryption, and digital signatures. 
Module 3: Consensus Algorithms — Understanding Proof of Work, Proof of Stake, and 
Byzantine Fault Tolerance. 
Module 4: Smart Contracts and Ethereum — Writing and deploying smart contracts using 
Solidity and Remix IDE. 
Module 5: Blockchain and AI Integration — Using blockchain to secure AI model sharing 
and training data provenance.

Module 2: Cryptography Fundamentals — Basics of hashing, public and private key
